<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/13%EC%A3%BC%EC%B0%A8_%EB%B3%B5%EC%8A%B5%EA%B3%BC%EC%A0%9C_%EC%99%84%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13주차 복습 과제: RAG 고도화 실험 — 하루스테이 숙박 챗봇

## 목표
> 12주차 RAG 베이스에 **고도화 기법**을 하나씩 적용하며 효과를 확인합니다.

실습의 위니브마켓 문서를 **하루스테이 숙박 정책**으로 바꿔, 다양한 메뉴를 적용해봅니다.

| 파트 | 실험 | 내용 |
| --- | --- | --- |
| **A** | 베이스 RAG | 12주차 구조 재구성 |
| **B** | 청킹 전략 | chunk_size별 비교 |
| **C** | 하이브리드 검색 | BM25 + 벡터 |
| **D** | MMR (다양성) | lambda_mult 조절 |
| **E** | 메타데이터 필터 | 카테고리별 검색 |
| **F** | 프롬프트 고도화 | 환각 방지 + few-shot |
| **G** | 출처 표시 | 답변 + 근거 문서 반환 |

---
## 0. 환경 설정

In [ ]:
!pip install -q langchain langchain-google-genai langchain-chroma chromadb langchain-text-splitters langchain-community rank_bm25
print("✅ 설치 완료!")

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "googlekey"
print("✅ API 키 설정 완료!")

---
## Part A. 베이스 RAG 구성

### A-1. 하루스테이 숙박 정책 문서

In [ ]:
policy_text = """하루스테이 고객 정책 안내 (정책번호: HS-POL-2024)

[체크인·체크아웃]
체크인은 오후 3시부터 가능하며, 체크아웃은 오전 11시까지입니다.
얼리 체크인은 오후 1시부터 가능하며, 추가 요금 2만원이 부과됩니다.
레이트 체크아웃은 오후 1시까지 가능하며, 추가 요금 3만원이 부과됩니다.
사전 연락 없이 오후 6시 이후 미도착 시 노쇼로 처리될 수 있습니다.
체크인 시 신분증 확인이 필요합니다.

[예약 및 결제]
온라인 예약은 체크인 날짜 기준 최소 1일 전까지 가능합니다.
예약 시 신용카드 또는 계좌이체로 결제할 수 있습니다.
예약 확인은 마이페이지 또는 예약번호로 조회할 수 있습니다.
성수기(7~8월, 12~1월)에는 최소 2박 이상 예약이 필요합니다.
예약 문의: 고객센터 1577-8888

[취소 및 환불]
체크인 7일 전까지 취소 시 전액 환불됩니다.
체크인 3일 전까지 취소 시 숙박비의 70%가 환불됩니다.
체크인 1일 전 취소 시 숙박비의 50%가 환불됩니다.
체크인 당일 취소 및 노쇼는 환불이 불가합니다.
태풍, 폭설 등 천재지변 시에는 전액 환불이 적용됩니다.
환불은 취소 신청 후 영업일 기준 3~5일 내 처리됩니다.

[객실 이용 규정]
모든 객실에 무료 와이파이, 미니바, 커피머신, 금고가 비치되어 있습니다.
반려동물 동반 투숙은 펫 프렌들리 객실에서만 가능하며, 추가 요금 5만원이 부과됩니다.
엑스트라 베드는 1대당 3만원이며, 프론트에 사전 요청이 필요합니다.
객실 내 취사는 금지되어 있으며, 전기포트와 전자레인지는 사용 가능합니다.
흡연은 지정된 흡연 구역에서만 가능합니다. 객실 내 흡연 적발 시 청소비 10만원이 부과됩니다.

[부대시설]
조식 뷔페는 1층 레스토랑에서 오전 7시부터 10시까지 운영됩니다.
조식 가격은 성인 25,000원, 아동(만 12세 이하) 15,000원입니다.
수영장은 여름 시즌(6~9월)에 오전 9시부터 오후 6시까지 운영됩니다.
피트니스 센터는 투숙객에게 무료로 24시간 개방됩니다.
지하 주차장은 투숙객 무료이며, 외부 방문객은 시간당 3,000원입니다.
세탁 서비스는 오전 9시까지 맡기면 당일 오후 6시에 수령 가능합니다.
비즈니스 센터에서 프린트, 복사, 팩스 서비스를 이용할 수 있습니다.

[VIP 멤버십]
연간 10박 이상 투숙 시 VIP 등급이 부여됩니다.
VIP 회원은 객실 업그레이드, 레이트 체크아웃 무료, 조식 20% 할인 혜택을 받습니다.
VIP 전용 라운지는 3층에 위치하며, 오후 2시부터 10시까지 운영됩니다.
"""

print(f"📄 문서 길이: {len(policy_text)}자")

### A-2. 임베딩 & 벡터 DB + 헬퍼 함수

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import chromadb

embedding = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_text(policy_text)

def make_store(texts, name="temp"):
    client = chromadb.Client()
    return Chroma.from_texts(texts=texts, embedding=embedding, client=client, collection_name=name)

vectorstore = make_store(chunks, "base")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def show_docs(docs, n=70):
    print(f"  📄 검색된 청크 {len(docs)}개")
    for i, d in enumerate(docs):
        meta = f"   {d.metadata}" if d.metadata else ""
        preview = d.page_content[:n].replace("\n", " ").strip()
        print(f"   [{i}] {preview} ...{meta}")

def compare_search(question, retrievers, n=60):
    print(f"❓ 질문: {question}\n")
    for name, ret in retrievers.items():
        print(f"  🔎 {name}")
        show_docs(ret.invoke(question), n)
        print()

print(f"✅ 베이스 구성 완료! 청크 {len(chunks)}개")

### A-3. 프롬프트 · LLM · 기본 RAG 체인

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

basic_prompt = ChatPromptTemplate.from_template(
    "아래 [컨텍스트]에만 근거해 답하세요. 없으면 '모른다'고 답하세요.\n\n"
    "[컨텍스트]\n{context}\n\n[질문]\n{question}"
)

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | basic_prompt | llm | StrOutputParser()
)

def ask(question, chain=None):
    c = chain or rag_chain
    answer = c.invoke(question)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print()
    return answer

ask("체크인 시간이 몇 시인가요?")
ask("취소하면 환불 얼마나 되나요?")
print("✅ 베이스 RAG 동작 확인!")

---
## Part B. 실험 — 청킹 전략

In [ ]:
for size in [100, 300, 500, 800]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    cks = sp.split_text(policy_text)
    print(f"chunk_size={size:>4} → 청크 {len(cks)}개  (평균 {len(policy_text)//len(cks)}자)")

In [ ]:
size = 100
sp_small = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=20)
chunks_small = sp_small.split_text(policy_text)
retriever_small = make_store(chunks_small, f"chunk_{size}").as_retriever(search_kwargs={"k": 3})

compare_search(
    "취소 환불 규정 알려주세요",
    {"베이스 (300)": retriever, f"잘게 ({size})": retriever_small},
)

---
## Part C. 실험 — 하이브리드 검색 (BM25 + 벡터)

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

bm25 = BM25Retriever.from_texts(chunks)
bm25.k = 3

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25, retriever],
    weights=[0.4, 0.6],
)

print("✅ 하이브리드 검색기 준비 완료")

In [ ]:
compare_search(
    "HS-POL-2024",
    {"벡터만": retriever, "BM25만": bm25, "하이브리드": ensemble_retriever},
)
print()
compare_search(
    "1577-8888",
    {"벡터만": retriever, "BM25만": bm25, "하이브리드": ensemble_retriever},
)

---
## Part D. 실험 — MMR (다양성)

In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 8, "lambda_mult": 0.5},
)

compare_search(
    "하루스테이 정책 전체 알려줘",
    {"일반 검색": retriever, "MMR": mmr_retriever},
)

---
## Part E. 실험 — 메타데이터 필터

In [ ]:
from langchain_core.documents import Document

sections = [
    ("체크인", "체크인은 오후 3시부터 가능하며, 체크아웃은 오전 11시까지입니다."),
    ("체크인", "얼리 체크인은 오후 1시부터 가능하며, 추가 요금 2만원이 부과됩니다."),
    ("체크인", "레이트 체크아웃은 오후 1시까지 가능하며, 추가 요금 3만원이 부과됩니다."),
    ("예약", "온라인 예약은 체크인 날짜 기준 최소 1일 전까지 가능합니다."),
    ("예약", "성수기(7~8월, 12~1월)에는 최소 2박 이상 예약이 필요합니다."),
    ("취소/환불", "체크인 7일 전까지 취소 시 전액 환불됩니다."),
    ("취소/환불", "체크인 3일 전 취소 시 70%, 1일 전 50% 환불됩니다."),
    ("취소/환불", "체크인 당일 취소 및 노쇼는 환불이 불가합니다."),
    ("취소/환불", "태풍, 폭설 등 천재지변 시에는 전액 환불이 적용됩니다."),
    ("객실", "반려동물 동반 투숙은 펫 프렌들리 객실에서만 가능하며, 추가 요금 5만원입니다."),
    ("객실", "객실 내 흡연 적발 시 청소비 10만원이 부과됩니다."),
    ("객실", "엑스트라 베드는 1대당 3만원이며, 사전 요청이 필요합니다."),
    ("부대시설", "조식 뷔페는 오전 7시~10시, 성인 25,000원, 아동 15,000원입니다."),
    ("부대시설", "피트니스 센터는 투숙객 무료로 24시간 개방됩니다."),
    ("부대시설", "지하 주차장은 투숙객 무료이며, 외부 방문객은 시간당 3,000원입니다."),
    ("부대시설", "수영장은 여름 시즌(6~9월) 오전 9시~오후 6시 운영됩니다."),
    ("VIP", "연간 10박 이상 투숙 시 VIP 등급이 부여됩니다."),
    ("VIP", "VIP 회원은 객실 업그레이드, 레이트 체크아웃 무료, 조식 20% 할인 혜택을 받습니다."),
]

docs_meta = [Document(page_content=text, metadata={"category": cat}) for cat, text in sections]
vectorstore_meta = Chroma.from_documents(docs_meta, embedding=embedding)

retriever_all = vectorstore_meta.as_retriever(search_kwargs={"k": 3})
retriever_cancel = vectorstore_meta.as_retriever(search_kwargs={"k": 3, "filter": {"category": "취소/환불"}})

compare_search(
    "환불 얼마나 되나요?",
    {"필터 없음": retriever_all, "취소/환불만": retriever_cancel},
)

---
## Part F. 실험 — 프롬프트 고도화 (환각 방지)

In [ ]:
improved_prompt = ChatPromptTemplate.from_template(
    "당신은 하루스테이 고객센터 상담원입니다.\n"
    "규칙:\n"
    "1) 반드시 아래 [컨텍스트]에 있는 내용만으로 답하세요.\n"
    "2) 컨텍스트에 없으면 추측하지 말고 '해당 내용은 안내되어 있지 않습니다'라고 답하세요.\n"
    "3) 답변은 3문장 이내로 간결하게.\n\n"
    "[예시]\n"
    "질문: 룸서비스 메뉴가 뭐가 있나요?\n"
    "답변: 죄송하지만 제공된 문서에는 룸서비스 메뉴가 안내되어 있지 않습니다.\n\n"
    "[컨텍스트]\n{context}\n\n[질문]\n{question}"
)

advanced_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | improved_prompt | llm | StrOutputParser()
)

print("=== 문서에 있는 질문 ===")
ask("조식 가격이 얼마예요?", advanced_chain)

print("=== 문서에 없는 질문 ===")
ask("공항 셔틀 운행 시간 알려주세요", advanced_chain)
ask("룸서비스 메뉴가 뭐가 있나요?", advanced_chain)

---
## Part G. 실험 — 출처 표시

In [ ]:
from langchain_core.runnables import RunnableParallel

retriever_meta2 = vectorstore_meta.as_retriever(search_kwargs={"k": 2})

answer_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | improved_prompt | llm | StrOutputParser()
)

rag_with_source = RunnableParallel(
    context=retriever_meta2,
    question=RunnablePassthrough(),
).assign(answer=answer_chain)

result = rag_with_source.invoke("취소하면 환불 얼마나 되나요?")

print(f"Q: {result['question']}")
print(f"A: {result['answer']}")
print()
print("📎 출처:")
for doc in result["context"]:
    print(f"  [{doc.metadata['category']}] {doc.page_content[:60]}...")

---
## 📝 회고

In [ ]:
review = """
[13주차 RAG 고도화 복습 정리]

Part A. 베이스 RAG — 12주차 구조를 하루스테이 문서로 재구성
Part B. 청킹 전략 — chunk_size 100/300/500/800 비교, 잘게 나눌수록 정밀하지만 문맥이 끊길 수 있음
Part C. 하이브리드 검색 — BM25+벡터로 고유명사(정책번호, 전화번호) 검색 개선
Part D. MMR — 비슷한 청크만 나오는 문제 해결, lambda_mult로 다양성 조절
Part E. 메타데이터 필터 — 카테고리별 검색 범위 제한
Part F. 프롬프트 고도화 — 추측 금지 + few-shot으로 환각 방지
Part G. 출처 표시 — 답변과 근거 문서를 함께 반환

[챗봇 구성 고민]
하루스테이 챗봇에 가장 적합한 조합:
- 하이브리드 검색 (예약번호, 전화번호 등 고유 코드 검색 필요)
- 메타데이터 필터 (카테고리별로 정확한 답변 범위 제한)
- 개선 프롬프트 (환각 방지 필수)
- 출처 표시 (고객에게 근거 제시)
"""
print(review)